# ClarifyRL — GRPO training (Qwen3-0.6B)

Train a small instruction-tuned LLM to **ask before guessing** instead of hallucinating, via **GRPO** with TRL's `environment_factory` pattern.

- **Env**: ClarifyRL HF Space (`agarwalanu3103/clarify-rl`) — accessed via WebSocket (HTTP `/step` is stateless on the Space).
- **Compute**: T4 16GB (Colab free tier) or HF Jobs `t4-small`.
- **Model**: `Qwen/Qwen3-0.6B` — TRL's documented tool-calling reference model (used in the `add_response_schema` docstring). Has a built-in response schema in TRL, fits comfortably on T4 (~10 GiB total footprint with full AdamW).
- **Time budget**: ~30–45 min for `MAX_STEPS=50` smoke run.

Episode shape per rollout:
```
system+user(prompt) → assistant tool_call ask_question → tool_response
                    → assistant tool_call ask_question → tool_response
                    → ... (≤ max_steps turns) ...
                    → assistant tool_call propose_plan → tool_response (terminal)
```
Reward = final rubric score (FormatCheck gate × weighted [FieldMatch · InfoGain · QuestionEfficiency · HallucinationCheck]).

## 1. Install dependencies

`environment_factory` (the new tool-calling API in `GRPOTrainer`) is experimental and TRL guards it with runtime soft-dependency checks instead of pulling everything via the `trl[vllm]` extra. As of TRL `main`, the gates we have to pass are:

| Gate | Required |
|---|---|
| `transformers >= 5.2.0` | install from `main` (not on PyPI yet) |
| `jmespath` | declared only in TRL's `[dev]` extras → install explicitly |
| Chat template supports tool calls | Qwen3 chat template is hard-coded into TRL's `add_response_schema` |
| `liger_kernel` (only if `use_liger_loss=True`) | N/A — we don't enable it |

The install is split into **four separate `pip` calls** so a slow CDN at any step (especially the ~440 MB `vllm` wheel) only costs that one chunk on retry — `--timeout=300 --retries=10` further hardens against transient timeouts:

1. **`vllm`** — heaviest wheel, isolated.
2. **`trl[vllm]` + extras** — `trl`, `trackio`, `websockets`, `datasets`, `jmespath`. `vllm` is already installed, so this is fast.
3. **`transformers @ main`** — overrides the stable version pulled in by `trl[vllm]` so `environment_factory` stops complaining.

`vllm` pins `transformers<5` in its metadata but works fine at runtime with 5.x — pip will print a yellow resolver warning, ignore it.

> **Colab gotcha:** if `transformers` was already imported in this kernel before the upgrade, click **Runtime → Restart session** after the install cell finishes, then re-run from cell 2. The version-check at the bottom of the install cell will tell you if a restart is needed.
>
> **If pip times out again,** just re-run the cell — completed installs are cached, only the failed wheel re-downloads.

In [ ]:
%pip install --timeout=300 --retries=10 -Uq vllm
%pip install --timeout=300 --retries=10 -Uq "trl[vllm]" trackio websockets datasets jmespath bitsandbytes
%pip install --timeout=300 --retries=10 -Uq --upgrade "git+https://github.com/huggingface/transformers.git@main"
%pip uninstall -y wandb -q

import os
os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import transformers
from packaging.version import Version

if Version(transformers.__version__) < Version("5.2.0"):
    raise RuntimeError(
        f"transformers {transformers.__version__} is too old for `environment_factory`. "
        "Restart the runtime (Runtime → Restart session) and re-run from this cell."
    )

import importlib.util

for _pkg in ("jmespath", "bitsandbytes"):
    if importlib.util.find_spec(_pkg) is None:
        raise RuntimeError(
            f"{_pkg} is not installed. Re-run this cell — it's not pulled in transitively by `trl[vllm]`."
        )

print(f"transformers {transformers.__version__} OK, jmespath OK, bitsandbytes OK")

## 2. Hugging Face login

Needed for `push_to_hub=True` and Trackio logging. The cell below resolves the token in this order — first hit wins, no popup if any of them succeed:

1. **Cached token** — if `~/.cache/huggingface/token` exists from a previous `notebook_login()` in this VM, just use it.
2. **`HF_TOKEN` env var** — if you exported it locally or in HF Jobs.
3. **Colab Secrets** — click the 🔑 key icon in the left sidebar, add a secret named `HF_TOKEN`, toggle "Notebook access" on. **One-time setup that survives runtime restarts forever.**
4. **Interactive popup** — last-resort fallback, you paste the token in the form.

> **Setting up Colab Secrets once is strictly less work than pasting the token after every Runtime → Restart.** Do it now, you'll thank yourself.

The token is **never** hardcoded in this notebook because cell `[24]` (`trainer.push_to_hub()`) uploads the entire notebook to the Hub — anything in the source becomes public.

In [ ]:
import os

from huggingface_hub import HfApi, login, notebook_login


def _whoami_or_none():
    try:
        return HfApi().whoami()
    except Exception:
        return None


_user = _whoami_or_none()

if _user is not None:
    print(f"Already logged in as {_user['name']!r} (cached token from earlier in this VM).")
else:
    _token = os.environ.get("HF_TOKEN")

    if _token is None:
        try:
            from google.colab import userdata  # type: ignore
            _token = userdata.get("HF_TOKEN")
        except Exception:
            _token = None

    if _token:
        login(token=_token, add_to_git_credential=False)
        os.environ["HF_TOKEN"] = _token
        _user = _whoami_or_none()
        print(
            f"Logged in via HF_TOKEN env/Colab-Secret as {_user['name']!r}"
            if _user
            else "Logged in via HF_TOKEN, but whoami() failed — token may be invalid."
        )
        del _token
    else:
        print(
            "No cached token, no HF_TOKEN env var, no HF_TOKEN Colab Secret. "
            "Falling back to interactive login — paste your token below.\n"
            "TIP: set up Colab Secrets (🔑 key icon → add HF_TOKEN) so you never see this popup again."
        )
        notebook_login()

## 3. Configuration

Conservative defaults: small model + short rollouts + 50 training steps so the first run completes in ~30 minutes on a single T4. Bump `MAX_STEPS` once the smoke run looks healthy.

In [ ]:
import os
import random

ENV_BASE_URL = os.environ.get("ENV_BASE_URL", "https://agarwalanu3103-clarify-rl.hf.space")
MODEL_NAME = "Qwen/Qwen3-0.6B"
OUTPUT_DIR = "clarify-rl-grpo-qwen3-0.6b"
MAX_STEPS = 50
SEED = 42

DIFFICULTIES = ["easy", "medium", "hard"]
DIFFICULTY_WEIGHTS = [0.5, 0.3, 0.2]

random.seed(SEED)

PROMPT = """You are a helpful assistant that books and plans things for users.
The user's request will be intentionally ambiguous — you do NOT yet have all the information needed to make a good plan.

You have three tools:
  - ask_question(question): ask the user ONE targeted clarifying question (max 6 across the episode).
  - propose_plan(plan): submit your final plan as a JSON STRING with the required fields. This ENDS the episode.
  - get_task_info(): re-read the original user request.

Strategy:
  1. Identify which fields the user has NOT specified.
  2. Use ask_question, ONE question per turn, to fill in just those fields.
  3. When you have enough info, call propose_plan with a JSON string.

Rules:
  - Be efficient. Each unnecessary question costs reward.
  - NEVER include fields in your plan that you weren't told about. No hallucinating values.
  - The `plan` argument MUST be a JSON STRING (not a dict). Example: propose_plan(plan='{\"start_time\": \"2pm\", \"duration\": \"30min\"}').
"""

print(f"Env:        {ENV_BASE_URL}")
print(f"Model:      {MODEL_NAME}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Max steps:  {MAX_STEPS}")
print(f"Seed:       {SEED}")

## 4. ClarifyEnv wrapper (WebSocket)

TRL's `environment_factory` instantiates this once per rollout and treats every public method with a docstring as a tool the model can call. The class talks to the OpenEnv MCP server over **WebSocket** (the HTTP `/step` path on the live Space does not share session state with `/reset`).

**Capacity-aware reset.** The deployed Space is single-tenant (`max_sessions=1`), but TRL launches `num_generations` rollouts in parallel — they collide on the only session every step. Without a retry, the loser of every race condition dies with `CAPACITY_REACHED` and produces a dead episode (reward 0). `reset()` therefore retries up to 6 times with exponential backoff (0.5 → 16 s, ~32 s max) on either `CAPACITY_REACHED` or transient WebSocket close errors, which is enough for the winning rollout to finish a typical 3–6-turn episode and free the slot.

In [ ]:
import json
import time
from typing import Any, Optional

from websockets.sync.client import connect as _ws_connect


def _ws_url(base_url: str) -> str:
    return base_url.replace("https://", "wss://").replace("http://", "ws://").rstrip("/") + "/ws"


class ClarifyEnv:
    """ClarifyRL env wrapper for TRL's environment_factory.

    One instance per rollout. Holds a WebSocket connection that lives for the
    duration of the episode. After self.done becomes True, the connection is
    closed automatically and the instance can be discarded.
    """

    def __init__(self) -> None:
        self._ws_url = _ws_url(ENV_BASE_URL)
        self._ws = None
        self.reward: float = 0.0
        self.done: bool = False

    def _open(self, retries: int = 3) -> None:
        if self._ws is not None:
            return
        last_err: Optional[Exception] = None
        for attempt in range(retries):
            try:
                self._ws = _ws_connect(
                    self._ws_url, open_timeout=30, close_timeout=10, max_size=2**24
                )
                return
            except Exception as exc:
                last_err = exc
                time.sleep(0.5 * (attempt + 1))
        raise RuntimeError(f"Could not open WS to {self._ws_url}: {last_err}")

    def _close(self) -> None:
        if self._ws is not None:
            try:
                self._ws.close()
            except Exception:
                pass
            self._ws = None

    def _send(self, payload: dict, timeout: float = 30.0) -> dict:
        self._open()
        self._ws.send(json.dumps(payload))
        raw = self._ws.recv(timeout=timeout)
        return json.loads(raw)

    @staticmethod
    def _extract_tool_result(obs_result: Any) -> dict:
        """Pull the canonical dict out of an MCP CallToolResult."""
        if isinstance(obs_result, dict):
            if "structured_content" in obs_result and isinstance(
                obs_result["structured_content"], dict
            ):
                return obs_result["structured_content"]
            if "data" in obs_result and isinstance(obs_result["data"], dict):
                return obs_result["data"]
            content = obs_result.get("content")
            if isinstance(content, list) and content:
                txt = content[0].get("text", "")
                try:
                    parsed = json.loads(txt)
                    if isinstance(parsed, dict):
                        return parsed
                except json.JSONDecodeError:
                    pass
            return obs_result
        if isinstance(obs_result, str):
            try:
                parsed = json.loads(obs_result)
                if isinstance(parsed, dict):
                    return parsed
            except json.JSONDecodeError:
                pass
        return {}

    def reset(self, **kwargs) -> Optional[str]:
        """Reset the env, retrying on transient errors (single-tenant capacity / WS close).

        The HF Space env is `max_sessions=1` but TRL runs `num_generations` rollouts in
        parallel. Without this retry, the loser of every race condition gets a permanent
        ERROR for that step and produces a dead episode (reward 0). We back off
        exponentially to give the winner's session time to finish.
        """
        self._close()
        self.reward = 0.0
        self.done = False
        task_id = random.choices(DIFFICULTIES, weights=DIFFICULTY_WEIGHTS, k=1)[0]

        max_attempts = 6  # total wait if all fail: 0.5+1+2+4+8+16 = 31.5 s
        last_err: str = ""
        for attempt in range(max_attempts):
            backoff = 0.5 * (2 ** attempt)
            try:
                resp = self._send({"type": "reset", "data": {"task_id": task_id}})
            except Exception as exc:
                last_err = f"connection error: {exc}"
                self._close()
                if attempt < max_attempts - 1:
                    time.sleep(backoff)
                    continue
                break  # no more retries — fall through to error return

            if resp.get("type") == "error":
                err_data = resp.get("data", {}) or {}
                err_code = err_data.get("code") if isinstance(err_data, dict) else None
                if err_code == "CAPACITY_REACHED" and attempt < max_attempts - 1:
                    last_err = "capacity reached, retrying"
                    self._close()  # release the WS so server can free our slot
                    time.sleep(backoff)
                    continue
                # Non-retryable RPC error — fail fast
                self.done = True
                return f"ERROR resetting env: {err_data}"

            # Success
            data = resp.get("data", {})
            obs = data.get("observation", {}) or {}
            self.reward = float(data.get("reward") or 0.0)
            self.done = bool(data.get("done") or False)
            info = self._extract_tool_result(obs.get("result"))
            request_text = info.get("request", "")
            family = info.get("family", "")
            max_steps = info.get("max_steps", 10)
            questions_remaining = info.get("questions_remaining", 6)
            return (
                f"USER REQUEST: {request_text}\n"
                f"Task family: {family}\n"
                f"You have {max_steps} turns and may ask up to {questions_remaining} clarifying questions.\n"
                f"Use the tools to clarify what is missing, then call propose_plan with a JSON string."
            )

        # All retries exhausted
        self.done = True
        return f"ERROR resetting env after {max_attempts} attempts: {last_err}"

    def _step_raw(self, tool_name: str, arguments: dict) -> dict:
        if self.done:
            return {"error": "episode already ended"}
        action = {"type": "call_tool", "tool_name": tool_name, "arguments": arguments}
        try:
            resp = self._send({"type": "step", "data": action})
        except Exception as exc:
            self.done = True
            self._close()
            return {"error": f"WS step failed: {exc}"}
        if resp.get("type") == "error":
            self.done = True
            self._close()
            return {"error": str(resp.get("data"))}
        data = resp.get("data", {})
        obs = data.get("observation", {}) or {}
        self.reward = float(data.get("reward") or 0.0)
        self.done = bool(data.get("done") or False)
        result_dict = self._extract_tool_result(obs.get("result"))
        if self.done:
            self._close()
        return result_dict

    def ask_question(self, question: str) -> str:
        """Ask the user a single clarifying question to gather missing information.

        Args:
            question: The clarifying question to ask, in plain English.

        Returns:
            The user's answer plus how many questions you have left.
        """
        out = self._step_raw("ask_question", {"question": question})
        if "error" in out:
            return f"Error: {out['error']}"
        answer = out.get("answer", "")
        qr = out.get("questions_remaining", "?")
        revealed = out.get("field_revealed")
        suffix = f" [revealed field: {revealed}]" if revealed else ""
        return f'User answered: "{answer}" (questions remaining: {qr}){suffix}'

    def propose_plan(self, plan: str) -> str:
        """Submit your final plan. The plan must be a JSON STRING. Ends the episode.

        Args:
            plan: A JSON-encoded string with the plan fields. For example:
                '{"start_time": "2pm", "duration": "30min", "meeting_type": "video"}'.
                MUST be a string, not a dict.

        Returns:
            The final score and per-component breakdown.
        """
        if not isinstance(plan, str):
            try:
                plan = json.dumps(plan)
            except Exception:
                plan = str(plan)
        out = self._step_raw("propose_plan", {"plan": plan})
        if "error" in out:
            return f"Error submitting plan: {out['error']}"
        score = out.get("score", 0.0)
        breakdown = out.get("breakdown", {})
        parse_err = out.get("parse_error")
        parts = [f"Final score: {float(score):.3f}"]
        if parse_err:
            parts.append(f"plan parse error: {parse_err}")
        if breakdown:
            parts.append(f"breakdown: {json.dumps(breakdown)}")
        return " | ".join(parts)

    def get_task_info(self) -> str:
        """Re-read the original user request.

        Returns:
            The user's request, task family, and remaining question budget.
        """
        out = self._step_raw("get_task_info", {})
        if "error" in out:
            return f"Error: {out['error']}"
        return (
            f"Request: {out.get('request', '')}\n"
            f"Family: {out.get('family', '')}\n"
            f"Questions remaining: {out.get('questions_remaining', '?')}"
        )

    def __del__(self):
        try:
            self._close()
        except Exception:
            pass

## 5. Smoke test

Run a complete 3-action episode end-to-end (reset → ask_question → propose_plan) and assert that the final reward signal arrives. **If this fails, training will fail. Fix the env first, do not proceed.**

In [ ]:
def _smoke():
    env = ClarifyEnv()
    print("--- reset ---")
    obs = env.reset()
    print(obs)
    print(f"reward={env.reward} done={env.done}\n")
    assert not env.done, "reset should not end the episode"

    print("--- ask_question ---")
    print(env.ask_question("What is the most important constraint?"))
    print(f"reward={env.reward} done={env.done}\n")

    print("--- propose_plan ---")
    print(env.propose_plan(
        '{"start_time": "2pm", "duration": "30min", "meeting_type": "video"}'
    ))
    print(f"reward={env.reward} done={env.done}")
    assert env.done, "propose_plan should end the episode"
    print("\nSMOKE TEST PASSED")

_smoke()

## 6. Reward function

TRL hands `reward_funcs` the list of `ClarifyEnv` instances after each rollout completes. We just read `env.reward`, which the env updated on the terminal `propose_plan` step (or 0.0 if the model never proposed).

In [ ]:
def reward_func(environments, **kwargs) -> list[float]:
    return [float(env.reward) for env in environments]

## 7. Dataset

One row per rollout. The actual user request is generated by the env on `reset()`, so the prompt is just the static system instructions. We over-provision rows so we never run out before `MAX_STEPS`.

In [ ]:
from datasets import Dataset

NUM_TRAIN_EPISODES = 2000
dataset = Dataset.from_dict({
    "prompt": [[{"role": "user", "content": PROMPT}] for _ in range(NUM_TRAIN_EPISODES)]
})
print(f"Dataset: {len(dataset)} episodes (≈ {MAX_STEPS * 8} consumed at current config)")

## 8. GRPOConfig

vLLM colocated mode for fast generation on a single T4. `enable_thinking=False` turns off Qwen3's `<think>` block so the model emits tool calls directly without first reasoning out loud.

### Memory budget on T4 (16 GiB)

`vllm_mode="colocate"` does **not** share weights with the trainer — vLLM keeps its own copy in inference layout. With Qwen3-0.6B the math is comfortable:

Empirical numbers from a successful T4 init (Qwen3-0.6B is actually 1.5 GiB on disk, not the 1.2 GiB fp16 napkin estimate — they ship with bf16 weights + extra metadata):

| Item | Size |
|---|---|
| HF model (bf16) | 1.5 GiB |
| vLLM model copy (fp16) | 1.5 GiB |
| Gradients (bf16) | 1.5 GiB |
| **AdamW state (8-bit m+v via bitsandbytes)** | **1.5 GiB** (was 6.0 GiB with fp32 — caused step-1 OOM) |
| Activations (grad-ckpt, batch=1) | ~1.5 GiB |
| vLLM KV cache + buffers | ~1.0 GiB |
| Optimizer temp buffers (`sqrt(v_t)` etc.) | ~0.2 GiB |
| Fragmentation (~3% with `expandable_segments`) | ~0.4 GiB |
| **Total** | **~9.1 GiB** ✅ (~5 GiB headroom on 14.56 GiB T4) |

Two memory-critical settings:

1. **`optim="adamw_8bit"`** — uses `bitsandbytes` to store the AdamW first/second moments in int8 instead of fp32. Saves 4.5 GiB at zero quality loss for our scale. Without this, full-precision AdamW workspace OOMs on the very first optimizer step.
2. **`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`** (set in cell 2) — switches PyTorch's CUDA allocator to expandable segments, which prevents memory fragmentation that wastes hundreds of MiB on tight GPUs.

`vllm_gpu_memory_utilization=0.30` gives vLLM ~4.4 GiB total budget — plenty for the 1.5 GiB model copy plus KV cache.

If you scale up to 1.7B / 3B / 7B, additional escape valves: lower `vllm_gpu_memory_utilization` to 0.20, halve `max_completion_length`, or wrap with PEFT/LoRA (only adapters in the optimizer).

In [ ]:
from trl import GRPOConfig

grpo_config = GRPOConfig(
    num_train_epochs=1,
    learning_rate=1e-6,
    gradient_accumulation_steps=8,
    per_device_train_batch_size=1,
    warmup_steps=10,
    max_steps=MAX_STEPS,
    optim="adamw_8bit",
    max_grad_norm=1.0,
    seed=SEED,
    num_generations=2,
    max_completion_length=1024,
    log_completions=True,
    num_completions_to_print=2,
    chat_template_kwargs={"enable_thinking": False},
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.30,
    vllm_max_model_length=3072,
    output_dir=OUTPUT_DIR,
    report_to="trackio",
    trackio_space_id=OUTPUT_DIR,
    logging_steps=1,
    save_steps=25,
    save_total_limit=1,
    gradient_checkpointing=True,
    push_to_hub=True,
)
print("GRPOConfig OK")

## 9. Initialise trainer

Two important pieces of housekeeping before constructing the trainer:

1. **CUDA cleanup.** Each failed `GRPOTrainer(...)` call (e.g. when an `ImportError` raised after `Loading weights:` finished) leaves a *ghost* model copy on the GPU because the constructed-but-not-bound object skips `__del__`. After several retries you hit `Free memory ... 0.39/14.56 GiB` from vLLM on the *next* attempt. The block below collects garbage, empties CUDA's allocator cache, and **fails fast** with a clear restart instruction if free memory is below the threshold.
2. **`OutStream.fileno` patch.** vLLM's `suppress_stdout` calls `sys.stdout.fileno()`, which the Jupyter `OutStream` raises on by default. Patching it to delegate to `sys.__stdout__.fileno()` fixes the crash.

> **If you hit OOM later during `trainer.train()`** (unlikely with Qwen3-0.6B but possible with bigger batches): swap `optim="adamw_torch"` → `optim="adamw_8bit"` in cell 16 (cuts optimizer state by 4× via `bitsandbytes`), drop `vllm_gpu_memory_utilization` to `0.20`, lower `max_completion_length` to `768`, or set `use_vllm=False` to fall back to the slower HF generation backend.

In [ ]:
import gc
import sys

import torch

for _name in ("trainer",):
    if _name in globals():
        del globals()[_name]
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    _free_b, _total_b = torch.cuda.mem_get_info()
    free_gib = _free_b / 1024**3
    total_gib = _total_b / 1024**3
    print(f"GPU memory before trainer init: {free_gib:.2f} / {total_gib:.2f} GiB free")
    if free_gib < 3.0:
        raise RuntimeError(
            f"Only {free_gib:.2f} GiB free on the GPU — not enough to load Qwen3-0.6B "
            "plus the vLLM colocate KV cache. This usually means a previous failed "
            "`GRPOTrainer(...)` left state behind. Click Runtime → Restart session, "
            "then re-run from cell [2] (wheels are cached, so it'll be fast)."
        )
else:
    print("WARNING: no CUDA device — training will be unusably slow")

try:
    from ipykernel.iostream import OutStream
    OutStream.fileno = lambda self: sys.__stdout__.fileno()
except Exception:
    pass

from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=grpo_config,
    environment_factory=ClarifyEnv,
)
print("Trainer initialised")

## 10. GPU info

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
    max_mem = round(gpu.total_memory / 1024**3, 3)
    print(f"GPU: {gpu.name} | Total: {max_mem} GB | Reserved before train: {start_mem} GB")
else:
    print("WARNING: no CUDA device — training will be unusably slow")

## 11. Train

This is the long-running cell. Things to watch in the per-step logs:

| Signal | Healthy | Bad |
|---|---|---|
| `reward_func` | non-zero on most rows after step ~10–15 | flat 0.0 for the whole run |
| Prompt column | shows `USER REQUEST: ...` for almost every row | shows `ERROR resetting env` / `CAPACITY_REACHED` for most rows |
| `<tool_response>` | `Final score: 0.xxx \| breakdown: {...}` | `Error: episode already ended` |
| `Training Loss` | mostly non-zero, can be negative (advantage-weighted) | always exactly 0.0 |

If you see `ERROR resetting env (after 6 attempts)` repeatedly, the Space is genuinely overloaded — wait a minute or restart it. If reward stays at 0.0 with healthy resets, the format gate is the culprit: inspect a completion, then iterate on the system prompt or `FormatCheck` rubric.

The first cell auto-refreshes `reset()` on the live env instances so you can edit cell 8 and re-run cell 22 without rebuilding the trainer (saves ~2 min of model reload).

In [ ]:
if 'trainer' in globals() and getattr(trainer, 'environments', None):
    _live_cls = type(trainer.environments[0])
    if _live_cls is not ClarifyEnv:
        _live_cls.reset = ClarifyEnv.reset
        print(f"Refreshed {_live_cls.__name__}.reset on {len(trainer.environments)} live env instances")

trainer_stats = trainer.train()
print(trainer_stats)

## 12. Save and push to Hub

In [ ]:
trainer.save_model(OUTPUT_DIR)
trainer.push_to_hub()
print(f"Saved to {OUTPUT_DIR} and pushed to the Hub")